<a href="https://colab.research.google.com/github/SaadH-077/geoverdict/blob/main/notebooks/01_geometry_gauntlet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — The geometry gauntlet: real EUDR plots, real satellite imagery

**Question this notebook answers:** when land-parcel geometries arrive from
suppliers — messy, mixed-type, in the wrong coordinate system — can we
validate, repair, and stand behind each one? And can we *measure* how good the
repair is?

**Why this is chapter one.** An EUDR due-diligence pipeline begins where the
data begins: geolocation submitted by operators. Everything downstream —
satellite lookups, forest baselines, verdicts — silently inherits the quality
of these polygons. A verdict computed over a plot whose latitude and longitude
were swapped is a confident answer about the wrong hemisphere. So before any
forest analysis, the pipeline needs an intake gate that **diagnoses** (never
silently fixes), **repairs with an audit trail**, and whose repair quality is
**measured, not assumed**.

**This chapter has two parts:**

- **Part A — the real problem.** We load **50 real EUDR example plots** from
  the FAO / Forest Data Partnership *Whisp* project, put them **on real
  Sentinel-2 imagery**, and run the actual validate → repair pipeline on them.
  Real submission polygons, real defects, real pixels underneath.
- **Part B — measuring repair quality.** Real data has no ground truth for
  *"what shape did the farmer intend?"*, and you cannot score a repair without
  it. So we build a **controlled benchmark**: generate known-good plots in the
  project's study area, corrupt them with a documented taxonomy, and measure
  detection, repair IoU, and false-alarm rate. This benchmark also produces the
  plot portfolio that chapters 02–06 screen for deforestation.

**Produces**
- `outputs/plots_analysis.geojson` — validated + repaired study-area portfolio (chapters 02–05 screen this)
- `outputs/whisp_validation.csv` — real-plot validation results
- `outputs/validation_report.csv`, `outputs/plots_truth.geojson`, `outputs/plots_submitted.geojson`
- `figures/g01_*.png`

**Runtime:** ~6 minutes. CPU only for the logic; the imagery cells make a few
dozen windowed reads against the public Sentinel-2 archive (no GPU, no login).

### 📦 Where the data in this notebook comes from

| Data | Real or synthetic? | Source | How it enters the notebook |
|---|---|---|---|
| **50 EUDR example plots** (Part A) | **real** | [FAO / Forest Data Partnership *Whisp*](https://github.com/forestdatapartnership/whisp), `tests/fixtures/geojson_example.geojson` | downloaded live from GitHub via `urllib` |
| **Sentinel-2 L2A imagery** under those plots (Part A) | **real** | Copernicus / ESA via [AWS Earth Search STAC](https://earth-search.aws.element84.com/v1) (Cloud-Optimised GeoTIFFs, no login) | windowed COG reads through `geoverdict.s2.basemap_rgb` |
| **Study-area plot portfolio** (Part B) | **synthetic** | generated in-notebook by `corrupt.generate_portfolio(...)` | created in memory |
| **The "damaged submission"** (Part B) | **synthetic** | `corrupt.corrupt_portfolio(...)` injects the failure taxonomy | created in memory |

**Why synthetic in Part B — and only there?** There is no public registry of
farm-plot boundaries for this deforestation frontier, and to *score* a repair
you must know the correct answer. So Part B generates known-good plots (the
answer key), damages them on purpose, and measures recovery. From **chapter 02
onward every value is measured from real Sentinel-2 imagery and real forest
maps** over real land.

### 🔌 What actually *is* the input? (the input contract)

"A supplier sends geometries" hides a lot. GeoVerdict's input is a **plot
portfolio**: per-plot records, each with a geometry, an optional declared area,
and a shared **sourcing region (AOI)**. Handling every variation below *is* the
job of this chapter:

| Input variation | EUDR / GeoJSON expectation | How GeoVerdict handles it |
|---|---|---|
| **Polygon / MultiPolygon** | norm for plots **> 4 ha** (EUDR Art. 9) | validated and repaired directly |
| **A single point** | allowed for plots **≤ 4 ha** | accepted (`POINT_GEOMETRY`) and **buffered to a footprint**; a point with declared area **> 4 ha** is refused (`POINT_TOO_LARGE`) |
| **Wrong axis order** (lat,lon) | GeoJSON mandates **lon,lat** | detected as `SWAPPED_AXES` against the AOI and swapped back |
| **Non-WGS84 CRS** (Web-Mercator, UTM, national grid) | GeoJSON mandates **EPSG:4326** | `LIKELY_PROJECTED`; Web-Mercator inverted + verified; unknown CRS → manual review |
| **Declared area / thresholds** | drives the 4 ha rule and sanity bounds | consumed as `declared_area_ha` |
| **Sourcing region / AOI** | the region plots should fall in | `config.AOI_BBOX` — makes `OUTSIDE_AOI` / `SWAPPED_AXES` detectable at all |
| **File formats** (GeoJSON, Shapefile, KML, WKT, CSV) | — | `geopandas`/`shapely` parse them into geometries; GeoVerdict works on geometries, not formats |

So *"is the input just polygons?"* — **no.** It is a portfolio of geometries
that may be polygons or points, in any CRS, correct or corrupted, and the
validator's contract is to turn that into a trustworthy footprint or an honest
refusal.

### Environment setup and persistence

On Colab this clones the repository, installs dependencies, and mounts Google
Drive so that **outputs survive the session**. Locally it is a no-op beyond
putting `src/` on the path.

**Why Drive.** A Colab VM is deleted when the session ends, and the notebooks
depend on each other's artefacts: 01 writes the validated plot portfolio that
every later chapter loads; 02 writes the forest baselines; 03 the time series;
04 the model predictions. `outputs/`, `figures/` and `evidence/` are therefore
redirected to Drive via environment variables that `geoverdict.config` reads
at import time — which is why they must be set *before* the import.

**Re-running this cell picks up code changes**: it hard-resets the clone to
`origin/main` and purges `geoverdict` from `sys.modules` (Python caches
imports; a `git pull` alone leaves the kernel running the old code). Treat the
clone as read-only — edit code locally and push, not inside `/content`.

In [ ]:
# --- edit these if you are running your own fork ---------------------------
GITHUB_USER = "SaadH-077"
USE_DRIVE = True          # False -> everything stays in the ephemeral session
# ---------------------------------------------------------------------------

import os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "geoverdict"

if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        PERSIST = Path("/content/drive/MyDrive/geoverdict")
        for sub in ("outputs", "figures", "outputs/evidence"):
            (PERSIST / sub).mkdir(parents=True, exist_ok=True)
        os.environ["GEOVERDICT_OUTPUT_DIR"] = str(PERSIST / "outputs")
        os.environ["GEOVERDICT_FIGURE_DIR"] = str(PERSIST / "figures")
        os.environ["GEOVERDICT_EVIDENCE_DIR"] = str(PERSIST / "outputs" / "evidence")
        print("persisting outputs and figures to", PERSIST)

    if not Path(REPO).exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        f"https://github.com/{GITHUB_USER}/{REPO}.git"], check=False)
    if Path(REPO).exists():
        os.chdir(REPO)
        subprocess.run(["git", "fetch", "--quiet", "--depth", "50", "origin", "main"], check=False)
        before = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
        subprocess.run(["git", "reset", "--hard", "--quiet", "origin/main"], check=False)
        after = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
        if before != after:
            print(f"repo updated {before[:7]} -> {after[:7]}")
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    except subprocess.CalledProcessError as exc:
        print("!! dependency install failed:", exc)
        print("!! continuing anyway - the cells below will report what is missing")

ROOT = Path.cwd() if Path.cwd().name == REPO or (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

# Purge cached geoverdict modules so a repo update takes effect in THIS kernel.
for m in [m for m in list(sys.modules) if m == "geoverdict" or m.startswith("geoverdict.")]:
    del sys.modules[m]

from geoverdict import config as cfg
from geoverdict import viz

cfg.ensure_dirs()
viz.set_style()
print(f"outputs -> {cfg.OUTPUT_DIR}")
print(f"figures -> {cfg.FIGURE_DIR}")
print(f"seed = {cfg.SEED}, AOI = {cfg.AOI_NAME} {cfg.AOI_BBOX}")

### The failure taxonomy

Each class the validator diagnoses, with severity. **ERROR** = the plot cannot
be analysed until fixed (a verdict would be meaningless). **WARNING** =
analysable, but recorded because it degrades confidence. Severity is a
*decision*: a clockwise ring violates the GeoJSON spec but every library still
computes its area correctly → WARNING. A self-intersection changes what area
even *means* → ERROR.

In [ ]:
import pandas as pd
from geoverdict import geometry as G

pd.DataFrame([(c, sev, desc) for c, (sev, desc) in G.TAXONOMY.items()],
             columns=["code", "severity", "description"]
             ).sort_values(["severity", "code"]).reset_index(drop=True)

---
## Part A — Real EUDR plots, on real satellite imagery

The [Whisp](https://github.com/forestdatapartnership/whisp) project is the
open-source reference tool for EUDR plot screening (FAO / Forest Data
Partnership). Its example file ships **50 real plots from actual submissions**
across Africa, South America and Southeast Asia — genuine EUDR geometries, not
anything we made. We load them straight from GitHub.

In [ ]:
import json, urllib.request
import numpy as np
from shapely.geometry import shape

WHISP_URL = ("https://raw.githubusercontent.com/forestdatapartnership/whisp/"
             "main/tests/fixtures/geojson_example.geojson")
with urllib.request.urlopen(WHISP_URL, timeout=60) as r:
    whisp_raw = json.load(r)

whisp = [shape(f["geometry"]) for f in whisp_raw["features"]]
whisp_areas = np.array([G.area_ha(g) for g in whisp])
print(f"loaded {len(whisp)} real EUDR plots")
print(f"geometry types: {pd.Series([g.geom_type for g in whisp]).value_counts().to_dict()}")
print(f"areas: median {np.median(whisp_areas):.1f} ha, "
      f"range {whisp_areas.min():.2f}-{whisp_areas.max():,.0f} ha")
cx = np.array([g.centroid.x for g in whisp]); cy = np.array([g.centroid.y for g in whisp])
print(f"spread across the world: lon {cx.min():.0f}..{cx.max():.0f}, lat {cy.min():.0f}..{cy.max():.0f}")

### Seeing them: real plots on real Sentinel-2

For a representative sample we fetch a true-colour Sentinel-2 basemap around
each plot (least-cloudy recent scene, windowed COG reads — the same public
archive the rest of the project uses) and draw the submitted boundary on top.
This is the view an analyst actually works from: *does this polygon fall on a
field, a forest, a river, the sea?* The imagery is display-only (a cosmetic
per-tile stretch); nothing here is measured.

In [ ]:
import matplotlib.pyplot as plt
from geoverdict import s2

# a spread of plots across continents, biggest first so the imagery is legible
order = np.argsort(-whisp_areas)
sample_idx = list(order[:9])

fig, axes = plt.subplots(3, 3, figsize=(13, 13))
covered = 0
for ax, i in zip(axes.ravel(), sample_idx):
    g = whisp[i]
    bbox = s2.geom_view_bbox(g)
    rgb, bbox, item_id, dt = s2.basemap_rgb(bbox, max_cloud=25, max_px=256)
    covered += rgb is not None
    lon, lat = g.centroid.x, g.centroid.y
    viz.plot_on_basemap(ax, rgb, bbox,
                        [(g, dict(edgecolor=viz.PALETTE["warn"], lw=2.0))],
                        title=f"plot {i} · {G.area_ha(g):.0f} ha · ({lon:.1f}, {lat:.1f})")
fig.suptitle("Real EUDR submission plots on real Sentinel-2 imagery (Whisp examples)",
             fontweight="bold", y=0.995)
fig.tight_layout()
viz.save(fig, "g01_whisp_on_imagery")
plt.show()
print(f"basemap coverage: {covered}/{len(sample_idx)} plots had a cloud-free recent scene")

### Validating the real plots

We run the full validator over all 50 (with the AOI check disabled — these
plots are global, so "outside the sourcing region" is meaningless here). This
is a genuine result, not a manufactured one: whatever the validator flags, it
found in real submission data.

In [ ]:
whisp_issues = G.validate_portfolio(whisp, aoi_bbox=None)

from collections import Counter
counts = Counter(i.code for plot in whisp_issues for i in plot)
n_clean = sum(1 for p in whisp_issues if not p)
n_error = sum(1 for p in whisp_issues if any(i.severity == G.ERROR for i in p))

rows = [{"plot": k, "area_ha": round(G.area_ha(whisp[k]), 2),
         "issues": "; ".join(i.code for i in p) or "clean"}
        for k, p in enumerate(whisp_issues)]
whisp_df = pd.DataFrame(rows)
whisp_df.to_csv(cfg.OUTPUT_DIR / "whisp_validation.csv", index=False)

print(f"{n_clean}/{len(whisp)} plots fully clean; {n_error} carry at least one ERROR")
print("\ndefects found in real EUDR plots:")
for code_, n in counts.most_common():
    print(f"  {code_:<20} {n:>3}  ({G.TAXONOMY[code_][0]})")
print("\nexamples of flagged real plots:")
print(whisp_df[whisp_df.issues != "clean"].head(8).to_string(index=False))

cfg.append_result({"notebook": "01", "name": "whisp_real_validation",
                   "n_plots": len(whisp), "n_clean": int(n_clean),
                   "n_with_error": int(n_error),
                   "defects": {k: int(v) for k, v in counts.items()}})

### Repair, verified against real satellite pixels

The genuinely-flagged real plots above are mostly *too-large / too-small*
cases, which correctly go to **manual review** — you cannot auto-invent the
right area, so there is no "after" to show for them. To demonstrate repair on
real imagery, we take three **clean** real plots and inject a *repairable*
error into each — a lat/lon **axis swap**, a **Web-Mercator CRS** mix-up, and a
**self-intersection** — then show **submitted (red) vs repaired (green)** over
the real Sentinel-2. Watch the green boundary snap back onto the actual field:
that is a repair you can verify against the ground, not just a metric. (The
corruptions are injected on purpose and labelled as such — the *plots and
imagery are real*.)

In [ ]:
from geoverdict import corrupt as _cor

# clean real plots that have usable imagery, small enough to see the field
clean_real = [k for k, p in enumerate(whisp_issues)
              if not p and 1.0 < G.area_ha(whisp[k]) < 200.0]
demo = clean_real[:3]
demos = [("swap_axes", "lat/lon swapped"), ("web_mercator", "wrong CRS (Web-Mercator)"),
         ("bowtie", "self-intersection")]

from shapely.geometry import MultiPolygon

def largest_poly(geom):
    "Real Whisp plots are MultiPolygons; some corruptions act on a single ring."
    if isinstance(geom, MultiPolygon):
        return max(geom.geoms, key=lambda p: p.area)
    return geom

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, k, (cname, label) in zip(np.atleast_1d(axes), demo, demos):
    g = largest_poly(whisp[k])
    damaged = _cor.CORRUPTIONS[cname][0](g, np.random.default_rng(cfg.SEED))
    # validate the damaged plot WITHOUT an AOI (these are global), then repair
    iss = G.validate_geometry(damaged, declared_area_ha=G.area_ha(g), aoi_bbox=None)
    # axis-swap/CRS need a location reference to fix — use each plot's own
    # bounds as a local AOI so the repair can verify it landed back
    local_aoi = g.buffer(0.5).bounds
    iss_local = G.validate_geometry(damaged, declared_area_ha=G.area_ha(g), aoi_bbox=local_aoi)
    res = G.repair(damaged, iss_local, declared_area_ha=G.area_ha(g), aoi_bbox=local_aoi)
    rgb, bbox, *_ = s2.basemap_rgb(s2.geom_view_bbox(g), max_cloud=25, max_px=256)
    styles = [(g, dict(edgecolor=viz.PALETTE["insufficient"], lw=2.4, alpha=0.5))]
    if res.ok:
        styles.append((res.geometry, dict(edgecolor=viz.PALETTE["forest"], lw=1.8)))
    viz.plot_on_basemap(ax, rgb, bbox, styles,
        title=f"plot {k}: {label}\n{'; '.join(res.actions) if res.ok else 'manual review'}")
    iou = G.iou(res.geometry, g) if res.ok else 0.0
    ax.set_xlabel(f"IoU vs original: {iou:.3f}" if res.ok else "-> manual review", fontsize=8)
    print(f"plot {k}: injected {cname} -> {'REPAIRED (IoU %.3f)' % iou if res.ok else 'manual review'}; "
          f"actions={res.actions}")
fig.suptitle("Repair on real imagery: grey = original field · injected error · green = repaired",
             y=1.03, fontweight="bold")
fig.tight_layout()
viz.save(fig, "g01_whisp_repair")
plt.show()

---
## Part B — Measuring repair quality (a controlled benchmark)

Part A proved the pipeline runs on real plots. But it **cannot tell us how
*good* the repairs are** — for a real self-intersecting polygon we do not know
what the farmer meant, so there is nothing to score against. That is a
fundamental limit of real data here, not a shortcut we are taking.

So, exactly as database fuzzers and data-quality suites do, we build a
controlled benchmark: take known-good geometry (the answer key), inject a
taxonomy of realistic corruptions with a seeded RNG, and measure (a) detection
per class and (b) repair quality as **IoU between the repaired polygon and the
known original**. Two honest caveats stated up front:

- **High detection is expected, not impressive.** We designed both the
  corruptions and the validator, so catching them mostly confirms the validator
  has no bugs. The *informative* numbers are **repair IoU**, the **honest-refusal
  behaviour**, and the **false-alarm rate on clean plots**.
- The plots here are **synthetic and placed in the project's study area** (Novo
  Progresso, Pará — the BR-163 "arc of deforestation"), because this same
  portfolio is what chapters 02–06 screen for real deforestation. Sizes follow a
  log-normal (median ≈ 5 ha), the standard model for landholding sizes.

In [ ]:
from geoverdict import corrupt

truth = corrupt.generate_portfolio(cfg.N_PLOTS, cfg.AOI_BBOX, seed=cfg.SEED)
areas = np.array([G.area_ha(p) for p in truth])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(areas, bins=np.geomspace(areas.min(), areas.max(), 40), color=viz.PALETTE["forest"])
axes[0].set_xscale("log"); axes[0].set_xlabel("plot area (ha, log)"); axes[0].set_ylabel("plots")
axes[0].axvline(4.0, color=viz.PALETTE["warn"], ls="--", lw=1.2)
axes[0].set_title(f"{cfg.N_PLOTS} study-area plots, median {np.median(areas):.1f} ha")

# ground the synthetic portfolio on a REAL basemap of the study area:
# markers are plot LOCATIONS (honest), drawn over real Sentinel-2 of the frontier
aoi_rgb, aoi_bbox, *_ = s2.basemap_rgb(cfg.AOI_BBOX, max_cloud=15, max_px=420)
if aoi_rgb is not None:
    axes[1].imshow(aoi_rgb, extent=[cfg.AOI_BBOX[0], cfg.AOI_BBOX[2], cfg.AOI_BBOX[1], cfg.AOI_BBOX[3]],
                   origin="upper")
axes[1].scatter([p.centroid.x for p in truth], [p.centroid.y for p in truth],
                s=5, color=viz.PALETTE["warn"], alpha=0.6)
axes[1].set_title("portfolio locations on real Sentinel-2\n(Novo Progresso, Pará — the arc of deforestation)")
axes[1].set_xticks([]); axes[1].set_yticks([]); axes[1].grid(False)
fig.tight_layout()
viz.save(fig, "g01_portfolio")
plt.show()
print(f"total portfolio area {areas.sum():,.0f} ha; plots <= 4 ha (points allowed): {(areas <= 4).sum()}")

### Injecting damage

45% of the portfolio is damaged — pessimistic on purpose, so every failure
class has enough samples for a stable rate. Types are assigned round-robin over
a seeded shuffle (a uniform draw would leave rare classes with too few
samples). Duplicates are *appended* as extra rows (duplication is a property of
the submission, not a geometry). Each plot carries a declared area with ±10%
supplier noise. The gallery shows one of each class: grey = intended, red = as
submitted, green = repaired.

In [ ]:
submitted, injected, declared = corrupt.corrupt_portfolio(truth, cfg.CORRUPTION_RATE, seed=cfg.SEED)
print(f"submission: {len(submitted)} rows ({len(submitted) - len(truth)} appended duplicates)")

showcase_plot = truth[int(np.argsort(areas)[len(truth)//2])]
rng = np.random.default_rng(cfg.SEED)
cases = []
for name, (fn, expected) in corrupt.CORRUPTIONS.items():
    damaged = fn(showcase_plot, rng)
    issues = G.validate_geometry(damaged, declared_area_ha=G.area_ha(showcase_plot))
    result = G.repair(damaged, issues, declared_area_ha=G.area_ha(showcase_plot))
    cases.append({"name": name, "clean": showcase_plot, "corrupted": damaged,
                  "repaired": result.geometry if result.ok else None,
                  "caption": (f"repaired, IoU {G.iou(result.geometry, showcase_plot):.2f}"
                              if result.ok else "-> manual review")})
fig = viz.corruption_gallery(cases, n_cols=4,
    title="grey = intended | red = as submitted | green = auto-repaired")
viz.save(fig, "g01_corruption_gallery")
plt.show()

### Before → after: repairs you can verify by eye

The most convincing evidence a repair worked is *seeing* it. For the
mechanically-repairable classes below, the left panel is the damaged submission
(red) over the faint intended shape (grey); the right panel is the repair
(green) over the same grey. When green lands on grey, the geometry is genuinely
back — and the IoU underneath puts a number on it.

In [ ]:
ba_classes = ["swap_axes", "web_mercator", "bowtie", "repeated_vertices", "collapse_to_point"]
rng2 = np.random.default_rng(cfg.SEED + 1)
ba_cases = []
for name in ba_classes:
    fn, _ = corrupt.CORRUPTIONS[name]
    dmg = fn(showcase_plot, rng2)
    iss = G.validate_geometry(dmg, declared_area_ha=G.area_ha(showcase_plot))
    res = G.repair(dmg, iss, declared_area_ha=G.area_ha(showcase_plot))
    ba_cases.append({"name": name, "clean": showcase_plot, "corrupted": dmg,
                     "repaired": res.geometry if res.ok else None,
                     "iou": G.iou(res.geometry, showcase_plot) if res.ok else None})
fig = viz.repair_before_after(ba_cases,
    title="grey = intended shape    ·    red = submitted    ·    green = repaired")
viz.save(fig, "g01_before_after")
plt.show()

### Detection: does the validator catch what we injected?

Reframed honestly (see caveats above): near-total detection here mainly
confirms the validator is bug-free. The two rows that *do* carry information:
**bow-tie < 100%** — a vertex swap that happens not to self-intersect produces
a *valid* polygon of the wrong shape, which no validator can catch (this is why
real workflows send the rendered plot back for visual confirmation); and the
**false-alarm rate on clean plots**, the operational cost of an over-eager gate.

In [ ]:
issues_per_plot = G.validate_portfolio(submitted, declared_areas=declared)

detected, false_alarms = {}, 0
for inj, issues in zip(injected, issues_per_plot):
    codes = {i.code for i in issues}
    if inj is None:
        if any(i.severity == G.ERROR for i in issues):
            false_alarms += 1
        continue
    detected.setdefault(inj, []).append(bool(set(corrupt.CORRUPTIONS[inj][1]) & codes))

rows = [{"corruption": n, "n": len(h), "detection_rate": float(np.mean(h)),
         "acceptable_codes": " | ".join(corrupt.CORRUPTIONS[n][1])}
        for n, h in detected.items()]
det = pd.DataFrame(rows).sort_values("detection_rate")
n_clean_total = sum(1 for x in injected if x is None)
print(det.to_string(index=False))
print(f"\nfalse-alarm rate on clean plots: {false_alarms}/{n_clean_total} = {false_alarms/n_clean_total:.1%}")

cfg.append_result({"notebook": "01", "name": "validator_detection",
                   "overall_detection_rate": float(np.mean([h for v in detected.values() for h in v])),
                   "false_alarm_rate": false_alarms / n_clean_total,
                   "bowtie_detection": float(np.mean(detected.get("bowtie", [1])))})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
colors = [viz.PALETTE["forest"] if r == 1 else (viz.PALETTE["warn"] if r >= .9 else viz.PALETTE["clearing"])
          for r in det["detection_rate"]]
ax.barh(det["corruption"], det["detection_rate"], color=colors)
ax.set_xlabel("detection rate"); ax.set_xlim(0, 1.05)
ax.set_title("Detection per injected class (near-100% is expected — see the text)")
for y, (r, n) in enumerate(zip(det["detection_rate"], det["n"])):
    ax.text(min(r + 0.01, 1.0), y, f"{r:.0%} (n={n})", va="center", fontsize=8)
viz.save(fig, "g01_detection_rates")
plt.show()

### Repair quality — the number that actually matters

Repair applies targeted fixes in a fixed order (frame → topology → size; the
order *is* the algorithm — `make_valid` on Web-Mercator metres is meaningless),
logs every action, and the **validator** re-checks the result: a repair counts
only if the diagnosis comes back clean. Quality per class = IoU vs the known
original. For unrecoverable classes (`teleport`, `micro`, `sliver`, `duplicate`)
the *correct* behaviour is to **refuse** and route to manual review — inventing
a plausible plot from unrecoverable input is a compliance liability. Those
refusals are the zero-IoU bars, and they are a feature.

In [ ]:
records, repaired_geoms = [], []
for k, (geom, inj, dec) in enumerate(zip(submitted, injected, declared)):
    issues = issues_per_plot[k]
    if not issues:
        records.append({"plot_id": k, "injected": inj, "status": "clean",
                        "actions": "", "unresolved": "", "iou_vs_truth": 1.0 if k < len(truth) else None})
        repaired_geoms.append(geom); continue
    result = G.repair(geom, issues, declared_area_ha=dec)
    tg = truth[k] if k < len(truth) else None
    q = G.iou(result.geometry, tg) if (result.ok and tg is not None) else None
    status = ("clean_warned" if not any(i.severity == G.ERROR for i in issues)
              else ("repaired" if result.ok else "manual_review"))
    records.append({"plot_id": k, "injected": inj, "status": status,
                    "actions": "; ".join(result.actions), "unresolved": "; ".join(result.unresolved),
                    "iou_vs_truth": q})
    repaired_geoms.append(result.geometry if result.ok else geom)

report = pd.DataFrame(records)
funnel = report["status"].value_counts()
print(funnel.to_string())
auto_ok = report["status"].isin(["repaired", "clean_warned"])
rq = (report[report["injected"].notna()].assign(auto=auto_ok)
      .groupby("injected").agg(n=("status", "size"), auto_resolved=("auto", "mean"),
                               mean_iou=("iou_vs_truth", "mean")))
print("\n" + rq.round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
order_s = ["clean", "clean_warned", "repaired", "manual_review"]
cols = [viz.PALETTE["forest"], viz.PALETTE["warn"], viz.PALETTE["accent"], viz.PALETTE["clearing"]]
vals = [int(funnel.get(s, 0)) for s in order_s]
axes[0].bar(order_s, vals, color=cols)
for x, v in enumerate(vals):
    axes[0].text(x, v, f"{v}\n({v/len(report):.0%})", ha="center", va="bottom", fontsize=9)
axes[0].set_title("The intake funnel: what happens to a submission"); axes[0].set_ylabel("plots")

rqp = rq.sort_values("mean_iou")
axes[1].barh(rqp.index, rqp["mean_iou"].fillna(0),
             color=[viz.PALETTE["forest"] if v >= .95 else (viz.PALETTE["warn"] if v >= .5 else viz.PALETTE["clearing"])
                    for v in rqp["mean_iou"].fillna(0)])
axes[1].set_xlabel("mean IoU of repaired vs intended"); axes[1].set_xlim(0, 1.05)
axes[1].set_title("Repair quality per class\n(zero bars = honest refusals -> manual review)")
fig.tight_layout()
viz.save(fig, "g01_repair_funnel")
plt.show()

analysable = auto_ok.mean() + (report.status == "clean").mean()
cfg.append_result({"notebook": "01", "name": "repair_funnel",
                   "n_submitted": len(report), "analysable_after_repair": float(analysable),
                   "manual_review": int(funnel.get("manual_review", 0)),
                   "mean_iou_repaired": float(report.loc[report.status == "repaired", "iou_vs_truth"].mean()),
                   "per_class_auto_resolved": rq["auto_resolved"].round(3).to_dict()})
print(f"analysable after auto-repair: {analysable:.1%}")

### Hand-off: the analysis set for chapters 02–06

Downstream chapters only see plots the gate passed. We save the truth (answer
key, for verification in chapter 06), the raw submission (so the gauntlet is
replayable), and **the analysis set**: repaired geometry plus each plot's area,
status, and any warnings that must be carried into the final verdict — an
overlap warning at intake is still a caveat on the compliance verdict three
chapters later, and that thread must not drop.

In [ ]:
import geopandas as gpd

def to_gdf(geoms, extra):
    return gpd.GeoDataFrame({**extra, "geometry": geoms}, crs="EPSG:4326")

to_gdf(truth, {"plot_id": [str(i) for i in range(len(truth))]}) \
    .to_file(cfg.OUTPUT_DIR / "plots_truth.geojson", driver="GeoJSON")
to_gdf(submitted, {"plot_id": [str(i) for i in range(len(submitted))],
                   "injected": [x or "" for x in injected], "declared_area_ha": declared}) \
    .to_file(cfg.OUTPUT_DIR / "plots_submitted.geojson", driver="GeoJSON")

mask = report["status"].isin(["clean", "clean_warned", "repaired"]).to_numpy()
warn = ["; ".join(sorted({i.code for i in issues_per_plot[k] if i.severity == G.WARNING}))
        for k in range(len(submitted))]
analysis = to_gdf(
    [g for g, m in zip(repaired_geoms, mask) if m],
    {"plot_id": [str(k) for k, m in enumerate(mask) if m],
     "area_ha": [G.area_ha(g) for g, m in zip(repaired_geoms, mask) if m],
     "geometry_status": report.loc[mask, "status"].tolist(),
     "geometry_warnings": [w for w, m in zip(warn, mask) if m]})
analysis.to_file(cfg.OUTPUT_DIR / "plots_analysis.geojson", driver="GeoJSON")
report.to_csv(cfg.OUTPUT_DIR / "validation_report.csv", index=False)
print(f"analysis set: {len(analysis)} plots -> plots_analysis.geojson")
analysis.head()

### What this chapter established

1. **The pipeline works on real EUDR plots.** 50 real Whisp submissions were
   validated on real Sentinel-2 imagery; the validator found genuine defects
   (the counts are in `whisp_validation.csv` and the ledger), and repairs were
   shown before/after on the pixels — an audit trail, not a black box.
2. **Repair quality is measured, not claimed.** The controlled benchmark gives
   a per-class IoU and an intake funnel (auto-analysable vs honest refusal);
   the exact numbers are in the ledger and re-verified in chapter 06. The
   informative results are the repair IoU, the refusals, and the **zero
   false-alarm rate** — not the near-100% detection, which is expected when you
   grade a validator on the taxonomy it was built for.
3. **The honest gaps are named**: valid-but-wrong-shape corruptions (some
   bow-ties) are undetectable in principle and need supplier confirmation; and
   the study-area portfolio is synthetic because scoring repairs requires
   ground truth.

**Next:** chapter 02 asks the question the whole regulation hinges on — *was
each of these plots forest on 31 December 2020?* — and gets two official,
disagreeing answers, from real forest maps.